# DDPM Part 3 of 3: one reverse sampling step

Part 3 of 3, after **u18-p04** and **u18-p05**: one step of DDPM ancestral sampling with the trained noise predictor.

```python
def p_sample_step(model, x_t, i, betas, z):
```

- `model(x, t)` returns the predicted noise $\varepsilon_\theta$ with the shape of `x`; `t` must be a long tensor of shape `(B,)`. Call it once, with every entry of `t` equal to `i`.
- `x_t`: the current batch, float tensor `(B, ...)`, at zero-based step `i` (math step $t = i + 1$).
- `i`: a Python int, $0 \le i < T$. `betas`: float64 tensor `(T,)`, with `alphas = 1 - betas` and `alpha_bars` the cumulative product, as in u18-p04.
- `z`: standard normal noise with the shape of `x_t`, supplied by the caller.

Return the sample at the previous step:

$$\mu = \frac{1}{\sqrt{\alpha_i}}\Big(x_t - \frac{\beta_i}{\sqrt{1 - \bar\alpha_i}}\,\varepsilon_\theta\Big), \qquad
x_\text{prev} = \begin{cases}\mu + \sqrt{\tilde\beta_i}\; z, & i > 0, \quad \tilde\beta_i = \dfrac{1 - \bar\alpha_{i-1}}{1 - \bar\alpha_i}\,\beta_i \\[6pt] \mu, & i = 0 \text{ (no noise on the last step).}\end{cases}$$

(All indices here are zero-based array indices.) Return a tensor with the shape and dtype of `x_t` that does **not** require grad, even when the model's parameters do: sampling must not build a graph. Do not modify the inputs.

**Example.** With `betas = torch.linspace(0.1, 0.3, 3, dtype=torch.float64)`, `model = lambda x, t: 0.5 * x`, `x_t = torch.tensor([[1.0, -2.0]], dtype=torch.float64)` and `z = torch.tensor([[0.3, 0.1]], dtype=torch.float64)`: step `i = 2` returns `[[1.0641, -1.8402]]` and step `i = 0` returns `[[0.8874, -1.7749]]`.

**Restrictions.** Only `torch`. No loops or comprehensions.

**Constraints.** $T \le 1000$, batches up to `(64, 3, 32, 32)`. Time limit 0.5 seconds per step with a small model.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem6_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem6_submission.py
import torch


def p_sample_step(model, x_t, i, betas, z):
    # Your code
    return x_prev


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem6_submission as submission
importlib.reload(submission)
# e.g. submission.p_sample_step(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9Wwtz2ziS/is4TdWGzMocWXncjjKeuzw8s9nKq5Js7V5ZLg4kQjLXFMghKD/i9X+/rxsACUqyY2fmLjtriSDQ7250N6CrQZ4NJmKw3v/LXjV6OhiKwUo1EkNX22+avCkUjb169eGt+CDrRjwS5UI8mohSK1GrM1UbJYxcVUWul8I0qqJ1hTKm1B5YMXrCaHK9bpTB6P4TPGb5YpHP10VziZHHGFC6qen7oEoZoEo9uFWZrfGo5YqJqepyVqjV09SsZ6vcmByoMEsWRXmusjRfVWXdEJ6jQVPW85PBMd7OpNaK+DtyT2lRlhXNauq1Il6VadImX6ly3aQ0Ph7RqFxaSEulZZ4Q0WuP0A1l1Sr1EmChEc5El+mylhnjNuW6njPpMpNVo7K98Wj8dO/jeO/DI1pRQbBjR8k1nhd5wXK6GpzIWkOWScWCmfL//qqKCmIXzYlshJOFIFw0aJkX0a+LulwJt9yPPvw1TqZ6qp+76SI3Yl7qBpL/fq3zxnz/I3389L2Diucqz3763s4GEYl43YhMLXJAFSQwsVjreQN5GEHayXgwfWhpa+Spwjc11Z2ihNWl+BVDvybi8J/P3354c/hJFLlpiCVlwRIwI8xJea5haeLjWovoLDc5qIqfiU8ErhH1GmjJBC95EVj7WebFusbKWuawS3oU53lzIqQAPCOXSpzXeQOGxaKsxQulJzDeS3FO5J5Lg7elXg6FJqgiW68qcZJnGabnulo3hqRntaCdSOdlddk+kPWQfN2jxvpLAai6otHvxMuykDPx8sPfjZA13IbstSZZaSLlgRFvx+JDXQ6FKUVRzmVBEGFVRqzWEMpMkVAhopI8J8frIic5zMD/CVS5kHPYXkKYxsmIlCvFck36jyxmA1iF2mtOaiUzYSoFhWFWXa6XJ8UlrKVYCNACKuZlTYKGg0PojcgXRCAe2bMcH85XSCifPhwevkp/fv7y8/uP4oCwE8NTPS8ksJMaosOLuarIUuLJVAv8q/DKzoJBQeIqi2CL2VCszNLPAV5dkpB15kboX6fciOZ2QFJdRRd+LYJJsMZriHwzGF2A/1ybRuq5ii6G9n3yWWlT1nGwnNGqZl1rcZFkiJfzkyhO5tUaf1nNUWwnK2ZTvGZ0h3Vd1gEUy3IAS1eJNLKu5SXo7vgwJ7JS0bJshjBKTX9hnwfTASwb8XI68JQZyLpZI1JGxDnmxwkvdbSwTDHHTyJY8VAspoMrAngNhRuLS1wZxB11Uak5opO4CuZfA11H2bwozW2UDUXdlMXBvtp7OhTSff2LpxeedQ6SW2KlaS7B6KIoJRFG44yz/yJgZmn5I5bO7dcb2LEMuOnxLt78K2avRQCF5AbhLW+wGnQURRQLRAqywvDduXsXoqdIKnOOhJrW5HrRAneGTPouCidECMOJi/44cdGf0PBUXUNiLAgibiUv6EPOTLQUe+I8jmOCDUbzL0qoAn4xIufb4Sp9OXGgE2eyQHwQUSHrJYVdRQYrrvAxSR4tr+Oe7tVva1ncwSr7TmWnEtE6Y0sfxixRECGbpnbvpwNWRwdkh7kEBtLN+T83i9A0mP7UyQHEbTlTKNUWAmllskkxjIGoZeY7GAjGV3j1H3WPNJqFoZ4ycpOyi7A66Ju5UR+MMNBIt2SLfhqFKBzgOEk550rTHjlLhD/SnMP6wE964KH2KU2NlhV28ub/NzQXSFDvGZSB0lvlBZkk7+2DGFtixt7bpyfK8jkESYlLHNLTZ65HHQGMMqWqg8/I9AIDcwR+hvg2ydsGEPe2EBpKCCa/u+gJHmqJ5FDMvl3u8k5yt1OsXzC+bxW7JLEzHNMPBQ6RtEhMNIt3xRos7+LMzuVtnJi5r6zbvmcTB0Mb7WDZOtSVh8MQguSlTFdrOAWym2ihh+Ihwik84+HD03P65imxqeNL7ABioRnzghJU4gE5DrLAJdxLaqSM9XK9QlKOLURUhZyrxGedvOuTPyEuHnWeJWPOZyUtIIzHdiayM9o8dLRJkH1NS3IYMhg2MS1VSGVULbG/fcmryK5hHD0D53jSGpcLIpcocWCLmaIEP1/klFlSytzx8t1VLv4s9q+fifOyPqWsXrLxClIf8tE2Xjohg/hOwpTE5zoyilJBUHVXKT8LRYzoGqbcOzJpLiGxHxrk3uCABMS57oN2JZJ3W0MRqvcAUVMJhgJpZYglzsPfyrmYAwEtAZM266ZqhRN7Sg5QXezN1hkiKQ1zDo60PKcCal3BnURzns8VAYSEcuzQ5cJ5MGj2EjVsQjSyQL5E2zdBwIo13iXiM73IawyDmeIBoaB9wSIjCausZ1bNiLJJQEhQXS5SBqNqH22+YkpZs3sxcpTGpSOO3wPhlCi+F2HZ0JlkSqZYkzdE49DuoESg+fHAQdoIRTPUNKfdkBwPxekY2PrxkUkebgz2OPm6KJhOEsSYxTAOhlkMqNiijLbYG8QR5pshQ+xHTVmy3aD4z5pJMl5cI8uHlalnrupqjZiNCW7HgMRaUz1/5UTj1nENicLOq3jHP0aJyhJl8AKmtaQE1haZV05PDIi841YHrfUS3qkyr7CuwIEms3IFaS8kzC9tJ065h7I2Uua2VaJ6XY5faIS7TEHzIOegcZIXGXndnDojKJhz8gWa+bdP79+BVs2eY5oMFLKvVpfNCUb2XguPR/zoWhxpltc/iR87HHiJZ3jlv3ms+Wmr5IezBeU/FHvRFPmsHbIfGEnWTV50w2X79V+m7JoElWxOwuUmX2rZLTOXZqO94B9q7AszOT9tR2DFmpoFttdg5jnY1KoNdxg4zZu9QmGa2E+eJvsupzMT6hOtqGc35JCC6sJtPj74VTKHlJdT7XEki7yARfvHaDoA2WWtqAJ0nRZkosnDv+tTTT0cUxbUUSm5AwCEeUVaSx66TPGvhx8P4TlOFskHfEZpSm2wNI0TBC5ajqyukjUom2pIJaHJCaxV1U00gjMgfSAwMQDu6n1xL0h8h736NzkRh49HY+6EvfnH8//5lL54/u7d4StQcDUdpK6BmKbEDBjz+rSP6kLN3Tfk+fbbvFxVoHU6uJ7qN+/ff0jfvX91+AngIthK8nMJX6Uv/4DhKvv1ubnU8/bFG+SQLwHDPn1SwcMrJJnd0y9K0/5c1ocXVRy2WD7bXszXuiypRKW3isjG1ivEG2xbqvVYLhU9oLAbQZnNPJ2fqPmpiUw9JxU3sr/ffrT+TtkwNYghx/LUCmdllhDMM4RFygBtj4/dxjQPWu1wMuRaibRRNXVu24q9LcpuQkfHu7LZplZI7FhKsBLU1yC0n4d+ukR9fsF5KDXl1HZ+eOTInsBaULS11E8oTBpe7krkUts4c6US+tTl9YS+YzaKn2tHoGtI05YAeSUIzPCTjS41yejo2BE6k+0475ON33K4HZquDQMLxmfwh9T2St04mThSRvhunsFVKBOrbaJgAQOCnR9RRUNJdz6jrjzYOeXcTRUJGbZDwD3yQOYr2azWRTiy4O4IQjXFZgj/XBanESljY+sO0nSabu3ZlgabVUWbytLEhMmdbG9grqWCfDmSPImS8hW1FuxjAmwQJGWO0QqZJ8IRFXMEfMXEWkXEk917Y6CLRGaZwxFvT+50E8xLTFXkpG7CebS3f7xjYae8ZF1llHXvWBsWicXtYvwZUW+TGygUumI5usY72O8caocseYUTJL7/DiluShDQNqQQiK4ngV0WsLH0JuF901Lm875Cf+59Z5P3DYNgOsjR7gT1XRCRbwOYZzdztXvSLozdZrWJkx0/kVWldBb5jo9dNB3YkGfD+3/15bULC++CL3L93u1iz9fL54b2oNj2VjYWJKWb91Y2b5Eybtm0jUH3JA6EBQYZgKQk0YG6Mfz7LMB6iTV5GmWHoHnJv0oqTznARgGeON4dXf5MOwoE40FdPRiKBxaI9yhywAfAh7p3+YBasde7+4m76P/MR5pb5BtPewfsu/YEcSa1IdnOT9wOwalgt0Fw+1lJFLH/5M/4ma1Gi3N5afbs8SoqGeSFeQuIjNJBK3VxOfQ4Ubl0W01krcA2kVwqRUnHpSjUEqX5OaVNgmcmSRIHmxmSMpGVpBQL+CTn+tJqIQo2zT/x5hjswpZct/nCDP8tomAv/ZPo5YWBCQHDvU0HFBjh5EPwIxPvsB1Ajn+PhnXpcbRHsp2WQXmf+dQ6UVpWNp/sd9tounWyDde7C8NUUcIu/lt42FSyBn4jIps5OQxHo+PQuHewfx8RdFhvY56D2xbLPPptDNs0iY29xyiGyY3JqGt1ojQXsW3uyKv+WP4dZ9e3VOqLOgeY4jLlTDYCCBS6KVVT/Yz+vVZ7TGemzLzOuaoQFUy1oZse7iSaJ+T++JoKNrironmGD+v7HaYZtVV83Zqgbsb3Jm1mkUrStH2Bks8lmwQcmeaCN/UF4QEIaGpB5afyOR9Vfi0LLis9p5YJNawoynJv0hkd/lI6FmTtvSFSBRsMzebjtI58J0w6MeETEhWcj1D2f33FWK9pvhf2Cpl41NZiiLSbVS5VsrJenh21KaJn5caZ4+OgJHaSogT8QLRTHvl0HXaPcWo4JEUpMxNFTMT3ZDbkE9yKGBA87FnUyYh8sEP5RBAdMeEELw3uJh6Ibb+wBePU3bWZ8s0a0z3axow1WVdJkyXSwLsSKgjhH3XQjomcG0rR1tEpgyUXNEeWqmPeLeiJveQ4TGd9SefzWcN9xy20YZTgrkXE4qTLISay08Nd3lqJb3kuQHReJp8aap+8fu+FZ5s8if2I3NOn1788f/Px7dBV6W4qwlnKzV1/ChzEsv6VJZLleORp6VfHfAmma1ZBnVleKzifVUYESjfTLE+janJCUXsyX39+/fbwY/rx8PmbYUvdRppjKkXW02+FJTSaUm/GdnaoO8lHJgFHwU0vaxpdYw5qCkLVBr71bBudg8UICXVEfzYXwmPsPHNEr7lgOLaGv80S+5CqE+oBpXYZRZ74GwQ36vcmXNul15Jwhuic49hGM8sjRWCOahxxm7I85TMFceX1MUlG1AH2octheSGNajtEtgfStcRevDkcjfY7Au7Fxk0Uu4MhJpNOYuiGWOkU5ZKgW/aj1q23QO8SlI8rhBf2TBbFJ/ERCvDHo9FocvzNjsxXE/t9HB4K73fYOf6s2J9ogpehfRV/XbmdrLJS2aTJXrbD7kIgrsO6/Z4sLO/nk9OBvfRHXthuGe09wJZvd4/w6763DJxveZMvWWhulrvu5xpbvsqzUygy+KuDPol3YY/2GkoaIhQqCxvY6RvF9jNZG48jyRu1MhHfntH9XYGjqj/3pwMUCTKiRRw2vBAl6BCSUxLCuJE883ZM1l+vEbesUVC2wrcItGdtI+BSeM71WoX6m9sN1sbDCQM5ejI5Rvzmc2FQ67rUgsPjdOBA+8kBNnpL3WAqe3aVKS7e2kMfGholo+v7nIltX3z4hl3n9+487nBuOyp/U0hzSjjqJEeeSon39t2NrQjeLba5h/VxDtZFqZf+XGYzZoso1/aWGWfzQyogFrIW/mQwDoOAQ86HGxs97d/JcEszpdc77qrcay/5o2gJ7xnweUV2xz2kg9UaOMGrYcVZdNNB7VA82rW9uXT22BdkdRvY7r0VtQtsrnpApWPU5a63ZaUcVGZlWURbhNl3BKrpWS6Ba3rg2hW+QLlxS+FrzQvhCx4b3dKU6hs6KJv44wEqd+zJbrtZ0Jmua5+h5j/pTjDtxaPwKdFsSlrffIY35PPuob2JOgyv3gzdLREitb1Xjp3ARXR1wT8qALf9gzN9eQhOtU7e8ibkAxIfl6Xkh2kaGVUshgJYP8OyVL/dbdYVWUzSTg5DIjO1kpruEhm+jmuPvtvFAEyFJzXpdfIGW72sI+DJ4gQWRPvO5mS1mtnJh6uZyjJkgtHn2+YDoXbHNR1jsIRzWWeOL7pct3XBJ2jKNhuXwGy7Lsn4tiDdLuaXMJ2nj+0rvkrZuKuUNCO6sA9Ho+NhPNwIzNMBbbLcTuCmoN026JxdSI6U2GIJL50r2ktb0Yth3Lug2XLqfbLxFwCDSS8gho6OrcO/yOsCxMLu+Qr2i6HY249jRBgv+wje0L6+aK9bt9cg1ELVioTGrEC2KaSXD8UM+YsZii9ezjNubHBBd2azlDPyTJ7mjw9pyj4i0Vnw3r+bBUdwFb7v+1u/7VQZ1q3i4YE4C27HzLygKicgK/PwlytR7xoynwIyTxGzZKcu1ggyNBCql9hl2ziwk0iFXg/UOKZzcSwBY7Oj/BiJJUWFxPxWNxFxK2k0Fg8JZ9x7K+lFd++PDGu0fYhLKHoNG8b55wBOiwYf+8eEo4/Yf34J9LrWNpW/Qa1DgSxRtTe37UF1eC2wkKtZJsnVZoDy5cuEdqik90un1mQsZJ4WO1St+ey+0gt8Gz5qb/YyUdeT7t4iz3H+1L/5bp2W+oyd29qxVr0bQHt3qruFO25VdxCub71EFMZpTuG8sxD3wgcauKipKPkdJfvIGJJH2Kf7Fsd+9fRx0BwrsLpTQUPJ7RNo+MLOIGv00K1soqMjeBS8f5yMjo9vg/5leyWTBOJuX8i3L8lRo/FQECOhv9mfCHzd6hZ0IYXlNSSHEFf5NRcCX49DO9dOMK+q1Vn4Q44f4g0FrfIscyZr7qAlgHhM4hgRm8h9bpPJsl3e3nTBzhrun/sbSvX7934L3edsoVLpDpqOnrChjPm/pYd/sLybdr8ZRKfnH374YSieEI37vZ850G8BxJ10TTIP9LyxZRdKW3XzVhiLn8SoyxKDN9TtJvfO3Y9Y4i3YE7FrS7Y/tbN9DuzFjb0jzQ2dHjHBj4TuaId93DtssPsx0f540x6RxzVsjSkCri5zo/5gq9y0tcd+yd3NdtwZ5XDLqJ4O6fevXzWn4bcssmhvCSej0MSmAxIm/4wXZSjIHMW3hJNRT4071t5blTb1qE7uFPxHHP3JF++lvUd2wZPbNPL4xi1leMd5YR6wU+7/2Zc7s81C2PhBGoID7a01fC1H3smpGR+qnCh/BuPfcTdu0v0amw/j6TegDDtaG7WZ4fV/4MUqoLvkf7D/aL72jZLlZanPxllkN2z8V0kuYA72b/UOK+RHY/r/phP03oUydz8YwDY/3M60hv1cAPRFFz7Tejwa7U63SAsbJQ+zScgRwtx396uHpTxTkPxVu+jaKeuZOFVwEKB6YKzIWAWD6+v/BZJYZ+Q=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
